# 🍛 DishRadar — dish-level early-warning for restaurant reviews
**BCHBA504 · Business Transformation using Analytics and AI · Group Capstone**

A restaurant's overall rating hides which dish is going wrong. DishRadar mines each review for *(dish or service moment, sentiment)* pairs, tracks every dish month by month, and fires an alert while the overall star rating still looks fine.

**How to run:** `Runtime → Run all`, then scroll to the dashboard at the bottom. Everything works offline with no API key.

| Rubric item | Where in this notebook |
|---|---|
| Problem + data | Section 1 (Spice Route Kitchen, 18 months of reviews) |
| Method + tool fit | Section 2 (aspect extraction) and Section 3 (alert rule) |
| Working model | Sections 2-4 |
| Results + limitations | Section 5 |
| Live demo | Section 6 dashboard: Time machine tab |

In [ ]:
%pip install -q vaderSentiment plotly ipywidgets

In [ ]:
import re, random, io, warnings
import numpy as np, pandas as pd
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import ipywidgets as w
from IPython.display import display, HTML, clear_output
from vaderSentiment.vaderSentiment import SentimentIntensityAnalyzer
warnings.filterwarnings("ignore")

SEED = 42
rng = np.random.default_rng(SEED); random.seed(SEED)

C = dict(ink="#0E2229", card="#15303A", line="#26474F", text="#EAF2F0", muted="#8FB0B3",
         good="#5FD6A0", warn="#F4B223", bad="#FF5A4E", info="#6EC1E4")

# Menu vocabulary: dish -> words guests use for it. Edit this for a different restaurant.
MENU = {
    "Chicken Dum Biryani": ["chicken dum biryani", "chicken biryani", "dum biryani", "biryani", "biriyani"],
    "Butter Chicken": ["butter chicken"],
    "Paneer Tikka": ["paneer tikka"],
    "Garlic Naan": ["garlic naan", "naan"],
    "Haleem": ["haleem"],
    "Gulab Jamun": ["gulab jamun"],
    "Mango Lassi": ["mango lassi", "lassi"],
    "Veg Manchurian": ["veg manchurian", "manchurian"],
}
# Service "moments" guests talk about
MOMENTS = {
    "Service speed": ["wait", "waited", "waiting", "slow", "quick", "prompt", "promptly", "service", "took forever", "delay"],
    "Staff behaviour": ["staff", "waiter", "waiters", "server", "manager", "host"],
    "Ambience": ["ambience", "ambiance", "music", "decor", "seating", "atmosphere", "noisy"],
    "Value for money": ["price", "priced", "overpriced", "pricey", "expensive", "value", "portion", "portions", "cost"],
}

def _rx(words):
    return re.compile(r"\b(" + "|".join(re.escape(x) for x in sorted(words, key=len, reverse=True)) + r")\b", re.I)

def set_menu(menu):
    global MENU, DISH_RX
    MENU = menu
    DISH_RX = {d: _rx(a) for d, a in MENU.items()}
set_menu(MENU)
MOMENT_RX = {m: _rx(a) for m, a in MOMENTS.items()}

# Teach VADER the food-and-service vocabulary it doesn't know
sia = SentimentIntensityAnalyzer()
sia.lexicon.update({
    "dry": -1.8, "bland": -2.0, "stale": -2.0, "cold": -1.3, "overcooked": -2.0, "undercooked": -2.0,
    "greasy": -1.8, "tasteless": -2.4, "soggy": -1.8, "rubbery": -1.8, "dropped": -1.2, "hardly": -0.9,
    "slow": -1.6, "forever": -1.0, "waited": -0.8, "noisy": -1.5, "cramped": -1.8, "overpriced": -2.2,
    "expensive": -1.2, "tiny": -1.0, "painfully": -1.0, "inattentive": -2.0,
    "flavourful": 2.0, "flavour": 1.0, "seasoned": 1.2, "authentic": 1.8, "juicy": 1.8, "aromatic": 2.0,
    "quick": 1.5, "prompt": 1.5, "promptly": 1.5, "attentive": 1.8, "reasonably": 1.0, "worth": 1.0,
})
print("Setup complete ✔")

Setup complete ✔


## 1 · The data
Real restaurant review data with dish-level history is hard to get, so this notebook **simulates 18 months of reviews (Jan 2025 – Jun 2026) for a fictional Hyderabad restaurant**. The simulation plants a known story: biryani quality starts slipping in Oct 2025, service speed slips later in 2026, and everything else stays steady. Because we know the truth, we can measure whether the tool finds it. To use real reviews, skip to the **Use your data** tab in Section 6.

*Disclose in your report: the data is synthetic, generated in this notebook.*

In [ ]:
POP = {"Chicken Dum Biryani": .42, "Butter Chicken": .30, "Garlic Naan": .22, "Paneer Tikka": .20, "Haleem": .12,
       "Gulab Jamun": .12, "Mango Lassi": .12, "Veg Manchurian": .05,
       "Service speed": .25, "Staff behaviour": .15, "Ambience": .12, "Value for money": .15}
BASE = {"Chicken Dum Biryani": .88, "Butter Chicken": .86, "Garlic Naan": .84, "Paneer Tikka": .85, "Haleem": .82,
        "Gulab Jamun": .87, "Mango Lassi": .74, "Veg Manchurian": .72, "Service speed": .80,
        "Staff behaviour": .85, "Ambience": .83, "Value for money": .78}

def p_pos(a, m):
    """Chance a mention of `a` in month index m is positive. The planted story lives here."""
    p = BASE[a]
    if a == "Chicken Dum Biryani" and m >= 9:  p -= .05 * (m - 8)
    if a == "Service speed" and m >= 13:       p -= .06 * (m - 12)
    if a == "Mango Lassi" and m >= 6:          p += .014 * (m - 5)
    return float(np.clip(p, .05, .97))

DISH_POS = ["The {d} was absolutely delicious and full of flavour.", "Loved the {d}, it was perfectly cooked and tasty.",
            "The {d} here is amazing, rich and well seasoned.", "Really enjoyed the {d}, it was fresh, hot and excellent."]
DISH_NEG = ["The {d} was dry and bland this time.", "Disappointed with the {d}, it was cold and stale.",
            "The {d} was overcooked and the quality has dropped.", "The {d} was greasy and tasteless, not worth it.",
            "Honestly the {d} was terrible and soggy."]
MEH = ["The {d} was okay, nothing special.", "The {d} was fine but not as good as before.", "The {d} was average, a bit of a letdown."]
BIRYANI_NEG = ["The {d} had hardly any chicken and the rice was dry.", "The {d} felt bland and the rice was undercooked."]
MOM_POS = {"Service speed": ["Service was quick and the food arrived promptly.", "Impressively prompt service, no waiting at all."],
           "Staff behaviour": ["The staff were warm, polite and attentive.", "The waiter was friendly and very helpful."],
           "Ambience": ["Lovely ambience with comfortable seating.", "Great atmosphere and pleasant decor."],
           "Value for money": ["Great value for money with generous portions.", "Reasonably priced and worth every rupee."]}
MOM_NEG = {"Service speed": ["We waited almost 40 minutes and the service was painfully slow.", "Terrible service, the food took forever to arrive."],
           "Staff behaviour": ["The waiter was rude and ignored us.", "Staff were inattentive and unhelpful."],
           "Ambience": ["The place was noisy and the seating was cramped.", "Ambience was dull and uncomfortable."],
           "Value for money": ["Overpriced for the portion size.", "Too expensive and the portions were tiny."]}
OPEN = ["We visited on a weekend with family.", "Went there for dinner with friends.", "Ordered in a group of four.", "Dropped by for lunch.", ""]
CLOSE = ["", "", "Will see about coming again.", "That's my experience."]

def make_reviews(n_months=18, start="2025-01-01", per_month=70):
    months = pd.date_range(start, periods=n_months, freq="MS")
    rows, truth, rid = [], [], 0
    for mi, mo in enumerate(months):
        for _ in range(int(rng.normal(per_month, 8))):
            asp = [a for a in POP if rng.random() < POP[a]] or [random.choice(list(POP))]
            random.shuffle(asp); asp = asp[:4]
            sents, ok_list = [], []
            for a in asp:
                ok = rng.random() < p_pos(a, mi)
                if a in MENU:
                    d = random.choice(MENU[a])
                    pool = DISH_POS if ok else DISH_NEG + (BIRYANI_NEG if a == "Chicken Dum Biryani" else [])
                    if not ok and rng.random() < .2: pool = MEH   # lukewarm comments are harder to score
                    s = random.choice(pool).format(d=d)
                else:
                    s = random.choice((MOM_POS if ok else MOM_NEG)[a])
                sents.append(s); ok_list.append(ok); truth.append(dict(review_id=rid, aspect=a, truth=int(ok)))
            rating = int(np.clip(round(1 + 4 * np.mean(ok_list) + rng.normal(0, .55)), 1, 5))
            text = " ".join(x for x in [random.choice(OPEN)] + sents + [random.choice(CLOSE)] if x)
            rows.append(dict(review_id=rid, date=mo + pd.Timedelta(days=int(rng.integers(0, 27))), rating=rating, text=text))
            rid += 1
    return pd.DataFrame(rows), pd.DataFrame(truth)

reviews, truth = make_reviews()
print(f"{len(reviews):,} reviews · {reviews.date.min():%b %Y} to {reviews.date.max():%b %Y} · mean rating {reviews.rating.mean():.2f}★")
reviews.sample(5, random_state=1)[["date", "rating", "text"]].style.set_properties(subset=["text"], **{"white-space": "pre-wrap", "text-align": "left"})

1,193 reviews · Jan 2025 to Jun 2026 · mean rating 4.15★


,date,rating,text
49,2025-01-09 00:00:00,4,"Ordered in a group of four. The garlic naan here is amazing, rich and well seasoned. Really enjoyed the lassi, it was fresh, hot and excellent. The gulab jamun was dry and bland this time. Service was quick and the food arrived promptly."
763,2025-12-26 00:00:00,3,Service was quick and the food arrived promptly. The biryani had hardly any chicken and the rice was dry. That's my experience.
472,2025-08-11 00:00:00,5,"The staff were warm, polite and attentive. Loved the dum biryani, it was perfectly cooked and tasty. That's my experience."
1015,2026-04-12 00:00:00,1,"We visited on a weekend with family. Disappointed with the paneer tikka, it was cold and stale."
802,2026-01-10 00:00:00,4,"Dropped by for lunch. Really enjoyed the lassi, it was fresh, hot and excellent. The paneer tikka here is amazing, rich and well seasoned. Impressively prompt service, no waiting at all. The waiter was rude and ignored us."


## 2 · Extracting (dish, sentiment) pairs
**Default engine (offline):** each review is split into clauses; a clause that names a dish or service moment is scored with VADER plus a food-and-service lexicon added above.

**Optional upgrade:** `extract_llm()` does the same job with Claude, which handles sarcasm and unseen dishes better. It needs an Anthropic API key and was not run for the numbers in this notebook.

In [ ]:
def clauses(text):
    parts = re.split(r"(?<=[.!?])\s+|;|\b(?:but|however|although|though|whereas)\b", text, flags=re.I)
    return [p.strip(" ,") for p in parts if p and p.strip(" ,")]

def extract_lexicon(text):
    """Return [(aspect, kind, sentiment -1..1, clause)] for one review."""
    out = []
    for c in clauses(text):
        score = sia.polarity_scores(c)["compound"]
        for d, rx in DISH_RX.items():
            if rx.search(c): out.append((d, "Dish", score, c))
        for m, rx in MOMENT_RX.items():
            if rx.search(c): out.append((m, "Moment", score, c))
    return out

def extract_llm(texts, model="claude-sonnet-5-5", batch=20):
    """Optional: Claude-based extraction. Needs `%pip install anthropic` and ANTHROPIC_API_KEY.
    Returns rows shaped like extract_lexicon, keyed by review position."""
    import anthropic, json
    client = anthropic.Anthropic()
    out = {}
    for s in range(0, len(texts), batch):
        chunk = texts[s:s + batch]
        prompt = ("For each numbered restaurant review, list every dish and service moment mentioned "
                  "(service speed, staff behaviour, ambience, value for money) with sentiment from -1 to 1 and the evidence clause. "
                  'Reply with JSON only: [{"i":0,"aspect":"Chicken Biryani","sentiment":-0.7,"evidence":"..."}].\n\n'
                  + "\n".join(f"{i}. {t}" for i, t in enumerate(chunk)))
        msg = client.messages.create(model=model, max_tokens=4000, messages=[{"role": "user", "content": prompt}])
        for r in json.loads(msg.content[0].text.strip().strip("`").removeprefix("json")):
            kind = "Moment" if r["aspect"] in MOMENTS else "Dish"
            out.setdefault(s + r["i"], []).append((r["aspect"], kind, float(r["sentiment"]), r["evidence"]))
    return out

def build_mentions(df, engine="lexicon"):
    ex = extract_llm(df.text.tolist()) if engine == "llm" else None
    rows = []
    for i, r in enumerate(df.itertuples()):
        items = ex.get(i, []) if ex is not None else extract_lexicon(r.text)
        for a, k, s, c in items:
            rows.append(dict(review_id=r.review_id, month=r.month, aspect=a, kind=k, sentiment=s, clause=c))
    return pd.DataFrame(rows)

reviews["month"] = reviews.date.dt.to_period("M").dt.to_timestamp()
mentions = build_mentions(reviews)
print(f"Extracted {len(mentions):,} (aspect, sentiment) pairs from {len(reviews):,} reviews")

# Because the data is simulated we know the true polarity, so we can grade the extractor honestly.
chk = mentions.merge(truth, on=["review_id", "aspect"])
acc = ((chk.sentiment > 0) == (chk.truth == 1)).mean()
print(f"Extraction check against ground truth: {acc:.1%} of {len(chk):,} mentions got the right polarity")

Extracted 2,717 (aspect, sentiment) pairs from 1,193 reviews
Extraction check against ground truth: 98.9% of 2,717 mentions got the right polarity


## 3 · The early-warning rule
For every dish, DishRadar builds a **monthly sentiment score** and compares recent months with that dish's own healthy baseline:

* **Baseline:** the first 6 months give an average and a normal month-to-month wobble (σ).
* **Alert 🔴:** the last 3 months sit more than *z* σ below baseline. *z* is the sensitivity slider.
* **Watch 🟡:** halfway there, or a steady downward slope.
* **Not enough data ⚪:** fewer than the minimum mentions per month. This stops one angry review from raising an alarm.

To prove the lead time, the rule is **replayed month by month** using only data available at that point, then compared with the month the *overall* star rating (3-month average) first fell by an amount a manager would notice.

In [ ]:
def monthly(m, min_n):
    g = m.groupby(["aspect", "month"]).sentiment.agg(["mean", "size"]).reset_index()
    full = pd.date_range(m.month.min(), m.month.max(), freq="MS")
    score = g.pivot(index="month", columns="aspect", values="mean").reindex(full)
    n = g.pivot(index="month", columns="aspect", values="size").reindex(full).fillna(0)
    return score.where(n >= min_n), n

def assess(s, z=1.5, B=6, k=3):
    blank = dict(status="Not enough data", z=np.nan, base=np.nan, sd=np.nan, recent=np.nan, drop=np.nan, slope=np.nan)
    base, rec = s.iloc[:B].dropna(), s.iloc[-k:].dropna()
    if len(s) < B + k or len(base) < 3 or len(rec) < 2: return blank
    mu, sd = base.mean(), max(base.std(ddof=1), .12)
    tail = s.iloc[-6:]; mask = tail.notna().values
    slope = np.polyfit(np.arange(len(tail))[mask], tail.values[mask], 1)[0] if mask.sum() >= 3 else 0.0
    drop = mu - rec.mean(); zs = drop / sd
    if zs >= z: st = "Alert"
    elif zs >= z * .5 or (slope < -.04 and zs > 0): st = "Watch"
    elif zs <= -z: st = "Improving"
    else: st = "Healthy"
    return dict(status=st, z=zs, base=mu, sd=sd, recent=rec.mean(), drop=drop, slope=slope)

def first_alert(s, z=1.5, B=6, k=3):
    for t in range(B + k, len(s) + 1):
        if assess(s.iloc[:t], z, B, k)["status"] == "Alert": return s.index[t - 1]
    return None

def rating_notice(stars, drop=.25, B=6):
    """First month the 3-month average rating is `drop` stars below its own baseline."""
    base, roll = stars.iloc[:B].mean(), stars.rolling(3).mean()
    late = roll.iloc[B:]; hit = late[late <= base - drop]
    return (hit.index[0] if len(hit) else None), base, roll

def months_between(a, b): return (b.year - a.year) * 12 + b.month - a.month

def analyse(m, df, z, min_n, drop):
    score, n = monthly(m, min_n)
    stars = df.groupby("month").rating.mean()
    notice, rbase, roll = rating_notice(stars, drop)
    kinds = m.drop_duplicates("aspect").set_index("aspect").kind
    rows = []
    for a in score.columns:
        r = assess(score[a], z); fa = first_alert(score[a], z)
        rows.append(dict(aspect=a, kind=kinds[a], **r, first_alert=fa, mentions=int(n[a].sum()),
                         lead=(months_between(fa, notice) if fa is not None and notice is not None else None)))
    return dict(score=score, n=n, stars=stars, roll=roll, rbase=rbase, notice=notice, rows=rows, z=z, drop=drop)

R = analyse(mentions, reviews, 1.5, 6, .3)
tbl = pd.DataFrame(R["rows"])
print("Overall rating: baseline %.2f★ · managers would 'notice' in %s" % (R["rbase"], R["notice"].strftime("%b %Y") if R["notice"] is not None else "never (rating never fell enough)"))
tbl[["aspect", "kind", "status", "base", "recent", "z", "first_alert", "lead", "mentions"]].round(2).sort_values("z", ascending=False)

Overall rating: baseline 4.31★ · managers would 'notice' in Mar 2026


,aspect,kind,status,base,recent,z,first_alert,lead,mentions
2,Chicken Dum Biryani,Dish,Alert,0.75,0.24,4.21,2026-01-01,2.0,512
8,Service speed,Moment,Alert,0.35,0.08,2.26,2026-05-01,-2.0,313
1,Butter Chicken,Dish,Watch,0.62,0.51,0.97,NaT,NaN,365
10,Value for money,Moment,Healthy,0.45,0.42,0.21,NaT,NaN,183
3,Garlic Naan,Dish,Healthy,0.64,0.62,0.20,NaT,NaN,282
4,Gulab Jamun,Dish,Healthy,0.69,0.64,0.20,NaT,NaN,168
7,Paneer Tikka,Dish,Healthy,0.60,0.63,-0.14,NaT,NaN,236
9,Staff behaviour,Moment,Healthy,0.51,0.55,-0.31,NaT,NaN,204
5,Haleem,Dish,Healthy,0.59,0.72,-0.73,NaT,NaN,123
6,Mango Lassi,Dish,Healthy,0.40,0.65,-1.01,NaT,NaN,157


## 4 · Dashboard components
Charts, cards and the helper functions the dashboard uses.

In [ ]:
CSS = """<style>
@import url('https://fonts.googleapis.com/css2?family=Fraunces:opsz,wght@9..144,600&family=Hanken+Grotesk:wght@400;600;700&display=swap');
.dr{font-family:'Hanken Grotesk',system-ui,sans-serif;color:#EAF2F0;background:#0E2229;border-radius:16px;padding:14px}
.dr h1,.dr h2{font-family:'Fraunces',Georgia,serif;font-weight:600;margin:0}
.dr .hero{border-radius:14px;padding:22px 26px;background:linear-gradient(120deg,#15303A 0%,#1d4350 60%,#4a3b1d 130%);border:1px solid #26474F}
.dr .hero h1{font-size:30px}.dr .hero p{margin:8px 0 0;color:#B9D3D4;font-size:14.5px;max-width:720px;line-height:1.5}
.dr .kpis{display:grid;grid-template-columns:repeat(auto-fit,minmax(165px,1fr));gap:10px;margin:12px 0}
.dr .kpi{background:#15303A;border:1px solid #26474F;border-radius:10px;padding:12px 14px}
.dr .kpi small{color:#8FB0B3;font-size:12px}.dr .kpi b{display:block;font-size:25px;margin-top:3px}
.dr table{width:100%;border-collapse:collapse;background:#15303A;font-size:13.5px;border-radius:10px;overflow:hidden}
.dr th{text-align:left;color:#8FB0B3;font-weight:600;padding:9px 11px;background:#122930;font-size:12.5px}
.dr td{padding:9px 11px;border-top:1px solid #1f3d45}
.dr .badge{padding:2px 10px;border-radius:99px;font-weight:700;font-size:12px;white-space:nowrap}
.dr .chip{display:inline-block;margin:3px 6px 3px 0;padding:4px 10px;border-radius:99px;background:#1d4350;font-size:12.5px}
.dr .quote{background:#15303A;border-left:4px solid #FF5A4E;padding:8px 12px;margin:6px 0;border-radius:6px;font-size:13.5px}
.dr .quote.pos{border-color:#5FD6A0}
.dr .banner{border-radius:10px;padding:12px 16px;margin:8px 0;font-size:14.5px;line-height:1.5;background:#15303A;border:1px solid #26474F}
.dr .muted{color:#8FB0B3;font-size:13px}
.dr .guide{background:#15303A;border:1px solid #26474F;border-left:5px solid #F4B223;border-radius:10px;padding:12px 16px;margin:10px 0;line-height:1.55;font-size:14px}
.dr .guide span{display:block;color:#B9D3D4}
</style>"""
STATUS_COL = {"Alert": C["bad"], "Watch": C["warn"], "Healthy": C["good"], "Improving": C["info"], "Not enough data": C["muted"]}
STATUS_ICON = {"Alert": "🔴", "Watch": "🟡", "Healthy": "🟢", "Improving": "🔵", "Not enough data": "⚪"}
ORDER = {"Alert": 0, "Watch": 1, "Healthy": 2, "Improving": 3, "Not enough data": 4}

GUIDE = {
 "radar": ("Radar: which dishes need attention today?",
           "each row is a dish or service moment. The status compares its last 3 months with its own first 6 months. On the heatmap, red means unhappy guests, green means happy, grey means too few mentions.",
           "start with the red rows, then read the suggested next step."),
 "deep": ("Dish deep-dive: why is this one flagged?",
          "the blue line is how guests feel each month. The dotted green line is its healthy level and the dotted red line is the alert line. Cross the red line and the alert fires. The bars count positive and negative mentions.",
          "read the complaint words and quotes below the chart to find what changed."),
 "time": ("Time machine: would this really have warned us early?",
          "drag the month or press ▶. You only see what a manager would have known at that point. Blue is the dish, amber is the overall rating.",
          "look for the red line (dish alert) appearing before the amber line (rating visibly drops). The gap is the head start."),
 "try": ("Try a review: see what one score hides.",
         "paste a review. A normal overall score blends everything. DishRadar splits it by dish and service moment.",
         "write one review with a good dish and a bad one, then compare the two views."),
 "data": ("Use your data: run this on real reviews.",
          "upload a CSV with date, text and rating columns, then list the dishes you want tracked (comma separated).",
          "press Analyse, then open the other tabs. The dashboard now shows your restaurant."),
}
def guide(k):
    t, r, d = GUIDE[k]
    return f'<div class="guide"><b>{t}</b><span><b>Read it:</b> {r}</span><span><b>Do this:</b> {d}</span></div>'

def mon(t): return t.strftime("%b %Y") if t is not None and not pd.isna(t) else "—"
def badge(st): return f'<span class="badge" style="background:{STATUS_COL[st]}22;color:{STATUS_COL[st]}">{STATUS_ICON[st]} {st}</span>'
def wrap(html): return HTML(CSS + f'<div class="dr">{html}</div>')

def spark(s, color, W=110, H=26):
    v = s.values.astype(float); idx = np.where(~np.isnan(v))[0]
    if len(idx) < 2: return ""
    lo, hi = np.nanmin(v), np.nanmax(v); sp = (hi - lo) or 1
    pts = " ".join(f"{i / (len(v) - 1) * W:.1f},{H - 2 - (v[i] - lo) / sp * (H - 4):.1f}" for i in idx)
    return f'<svg width="{W}" height="{H}"><polyline fill="none" stroke="{color}" stroke-width="2" points="{pts}"/></svg>'

def style_fig(fig, h=480):
    fig.update_layout(template="plotly_dark", paper_bgcolor=C["ink"], plot_bgcolor=C["card"], height=h,
                      font=dict(family="Hanken Grotesk, system-ui, sans-serif", color=C["text"]),
                      margin=dict(l=55, r=55, t=60, b=40), legend=dict(orientation="h", y=1.12, x=0))
    fig.update_xaxes(gridcolor=C["line"]); fig.update_yaxes(gridcolor=C["line"])
    return fig

def vline(fig, x, color, text, row=None, y=1.0):
    fig.add_shape(type="line", x0=x, x1=x, y0=0, y1=1, xref="x", yref="paper", line=dict(color=color, width=2, dash="dot"))
    fig.add_annotation(x=x, y=y, xref="x", yref="paper", text=text, showarrow=False, font=dict(color=color, size=12),
                       bgcolor=C["ink"], yanchor="bottom")

NEG_WORDS = ["dry", "bland", "cold", "stale", "overcooked", "undercooked", "greasy", "tasteless", "soggy", "rubbery",
             "terrible", "slow", "rude", "noisy", "overpriced", "expensive", "tiny", "hardly", "cramped", "inattentive"]

def fig_heat(R):
    sc = R["score"].T
    order = sorted(sc.index, key=lambda a: (-(sc.loc[a].dropna().iloc[:6].mean() - sc.loc[a].dropna().iloc[-3:].mean()) if sc.loc[a].notna().sum() > 4 else 0))
    sc = sc.loc[order]
    fig = go.Figure(go.Heatmap(z=sc.values, x=[mon(t) for t in sc.columns], y=sc.index, zmin=-.1, zmax=.7,
                               colorscale=[[0, C["bad"]], [.5, C["warn"]], [1, C["good"]]], xgap=2, ygap=2,
                               colorbar=dict(title="Sentiment"), hovertemplate="%{y}<br>%{x}: %{z:.2f}<extra></extra>"))
    fig.update_layout(title="Sentiment by dish and month (grey = too few mentions)")
    fig.update_yaxes(autorange="reversed")
    return style_fig(fig, 430)

def fig_deep(R, a):
    s, n = R["score"][a], R["n"][a]; r = assess(s, R["z"]); fa = first_alert(s, R["z"])
    ev = mentions[mentions.aspect == a]
    pos = ev[ev.sentiment > .05].groupby("month").size().reindex(s.index).fillna(0)
    neg = ev[ev.sentiment < -.05].groupby("month").size().reindex(s.index).fillna(0)
    fig = make_subplots(rows=2, cols=1, shared_xaxes=True, row_heights=[.72, .28], vertical_spacing=.06)
    fig.add_trace(go.Scatter(x=s.index, y=s.values, mode="lines+markers", name="Monthly sentiment",
                             line=dict(color=C["info"], width=3), marker=dict(size=8), connectgaps=False), 1, 1)
    sm = s.ewm(span=3, min_periods=1).mean()
    fig.add_trace(go.Scatter(x=s.index, y=sm.values, mode="lines", name="Smoothed trend", line=dict(color=C["warn"], width=2, dash="dash")), 1, 1)
    if not np.isnan(r["base"]):
        fig.add_hline(y=r["base"], line=dict(color=C["good"], dash="dot"), annotation_text="healthy baseline", row=1, col=1)
        fig.add_hline(y=r["base"] - R["z"] * r["sd"], line=dict(color=C["bad"], dash="dot"), annotation_text="alert line", row=1, col=1)
    if fa is not None: vline(fig, fa, C["bad"], "⚠ early warning fires")
    fig.add_trace(go.Bar(x=s.index, y=pos, name="Positive mentions", marker_color=C["good"]), 2, 1)
    fig.add_trace(go.Bar(x=s.index, y=-neg, name="Negative mentions", marker_color=C["bad"]), 2, 1)
    fig.update_layout(title=f"{a}: how guests feel, month by month", barmode="relative")
    fig.update_yaxes(title_text="Sentiment", row=1, col=1); fig.update_yaxes(title_text="Mentions", row=2, col=1)
    return style_fig(fig, 560)

def complaint_html(a, R):
    ev = mentions[(mentions.aspect == a) & (mentions.sentiment < -.05)]
    months = R["score"].index
    recent, early = ev[ev.month >= months[-3]], ev[ev.month < months[6]]
    def cnt(df):
        t = " ".join(df.clause.str.lower()); return {w_: len(re.findall(rf"\b{w_}\b", t)) for w_ in NEG_WORDS}
    cr, ce = cnt(recent), cnt(early)
    top = sorted([(k, v) for k, v in cr.items() if v], key=lambda x: -x[1])[:6]
    chips = "".join(f'<span class="chip"><b>{k}</b> ×{v} <span class="muted">(was ×{ce[k]})</span></span>' for k, v in top) or '<span class="muted">No complaint words in the last 3 months.</span>'
    q = "".join(f'<div class="quote">“{c}”</div>' for c in recent.sort_values("sentiment").clause.drop_duplicates().head(4)) or ""
    ok = early[early.sentiment > .3].clause.drop_duplicates().head(2)
    p = "".join(f'<div class="quote pos">“{c}”</div>' for c in ok)
    return (f'<h2 style="font-size:19px;margin:6px 0">What guests complain about now</h2><div>{chips}</div>'
            f'<div class="muted" style="margin-top:8px">Recent complaints</div>{q}<div class="muted" style="margin-top:8px">How they talked about it in the first 6 months</div>{p}')

def action_html(a, R):
    ev = mentions[(mentions.aspect == a["aspect"]) & (mentions.sentiment < -.05)]
    months = R["score"].index
    def cnt(d):
        t = " ".join(d.clause.str.lower()); return {w_: len(re.findall(rf"\b{w_}\b", t)) for w_ in NEG_WORDS}
    cr, ce = cnt(ev[ev.month >= months[-3]]), cnt(ev[ev.month < months[6]])
    top = sorted([(k, v) for k, v in cr.items() if v], key=lambda x: -x[1])[:3]
    words = ", ".join(f"“{k}” ({v} times, was {ce[k]})" for k, v in top)
    fix = "Check the recipe, the ingredient supplier and who is cooking it on busy shifts." if a["kind"] == "Dish" else "Check staffing and the rota at peak hours."
    return f'<div class="banner">🛠 <b>Suggested next step for {a["aspect"]}</b><br>{fix}' + (f" Guests now say: {words}." if words else "") + "</div>"

def radar_html(R):
    rows = sorted(R["rows"], key=lambda r: (ORDER[r["status"]], -(r["z"] if not np.isnan(r["z"]) else -9)))
    alerts = [r for r in rows if r["status"] == "Alert"]
    leads = [r["lead"] for r in alerts if r["lead"] is not None]
    now = R["roll"].dropna().iloc[-1]
    kp = [("Reviews analysed", f"{len(reviews):,}"), ("Dishes and moments tracked", str(sum(r["status"] != "Not enough data" for r in rows))),
          ("Active alerts", f'<span style="color:{C["bad"] if alerts else C["good"]}">{len(alerts)}</span>'),
          ("Biggest head start", f"{max(leads)} months" if leads and max(leads) > 0 else "—"),
          ("Overall rating now (3-mo)", f"{now:.2f}★ <small>vs {R['rbase']:.2f}★ baseline</small>")]
    k = "".join(f'<div class="kpi"><small>{a}</small><b>{b}</b></div>' for a, b in kp)
    body = ""
    for r in rows:
        d = "" if np.isnan(r["drop"]) else f'{-r["drop"]:+.2f}'
        body += (f'<tr><td><b>{r["aspect"]}</b><div class="muted">{r["kind"]}</div></td><td>{badge(r["status"])}</td>'
                 f'<td>{spark(R["score"][r["aspect"]], STATUS_COL[r["status"]])}</td>'
                 f'<td>{"" if np.isnan(r["base"]) else f"{r['base']:.2f}"}</td><td>{"" if np.isnan(r["recent"]) else f"{r['recent']:.2f}"}</td><td>{d}</td>'
                 f'<td>{mon(r["first_alert"])}</td><td>{r["mentions"]}</td></tr>')
    top = alerts[0] if alerts else None
    msg = (f'<div class="banner">🔴 <b>{top["aspect"]}</b> has been below its healthy range since <b>{mon(top["first_alert"])}</b>'
           + (f', {top["lead"]} months before the overall rating dipped enough to notice.' if top["lead"] is not None and top["lead"] > 0 else ".")
           + " Open the <b>Time machine</b> tab to replay it.</div>" + action_html(top, R)) if top else '<div class="banner">🟢 No dish is currently below its healthy range.</div>'
    return (f'<div class="hero"><h1>DishRadar · Spice Route Kitchen</h1><p>The overall rating says 4 stars and change. The radar below says which dish is quietly slipping, and since when.</p></div>'
            + guide("radar") + f'<div class="kpis">{k}</div>{msg}<table><tr><th>Dish or moment</th><th>Status</th><th>Trend</th><th>Baseline</th><th>Last 3 mo</th><th>Change</th><th>Alert since</th><th>Times mentioned</th></tr>{body}</table>')

def fig_proof(R, a, upto):
    s = R["score"][a].iloc[:upto + 1]; stars = R["stars"].iloc[:upto + 1]
    notice, _, roll = rating_notice(stars, R["drop"]); fa = first_alert(s, R["z"]); r = assess(s, R["z"])
    fig = make_subplots(specs=[[{"secondary_y": True}]])
    fig.add_trace(go.Scatter(x=s.index, y=s.values, name=f"{a} sentiment (left)", mode="lines+markers", line=dict(color=C["info"], width=3), marker=dict(size=8)), secondary_y=False)
    fig.add_trace(go.Scatter(x=roll.index, y=roll.values, name="Overall rating, 3-mo avg (right)", mode="lines+markers", line=dict(color=C["warn"], width=3), marker=dict(size=7)), secondary_y=True)
    full = R["score"].index
    fig.update_xaxes(range=[full[0] - pd.Timedelta(days=15), full[-1] + pd.Timedelta(days=15)])
    lo = np.nanmin(R["score"][a].values) - .1; hi = np.nanmax(R["score"][a].values) + .1
    fig.update_yaxes(range=[lo, hi], title_text="Dish sentiment", secondary_y=False)
    rl, rh = np.nanmin(R["roll"].values) - .15, np.nanmax(R["roll"].values) + .15
    fig.update_yaxes(range=[rl, rh], title_text="Stars", secondary_y=True, showgrid=False)
    if fa is not None: vline(fig, fa, C["bad"], f"🔴 dish alert · {mon(fa)}")
    if notice is not None: vline(fig, notice, C["warn"], f"⭐ rating visibly drops · {mon(notice)}", y=.93)
    fig.update_layout(title=f"What a manager would have seen as of {mon(s.index[-1])}")
    style_fig(fig, 470)
    cur = roll.dropna().iloc[-1] if roll.notna().any() else np.nan
    if upto + 1 < 9: msg = "Learning the baseline from the first six months of reviews."
    elif fa is None: msg = f"{STATUS_ICON[r['status']]} <b>{a}</b> status: {r['status']}. The overall rating is {cur:.2f}★."
    elif notice is None or notice > s.index[-1]:
        msg = (f"🔴 <b>Alert fired in {mon(fa)}.</b> The overall rating is still <b>{cur:.2f}★</b> and looks fine. "
               f"Without DishRadar, nobody would be looking at {a} yet.")
    else:
        msg = (f"🔴 Alert fired in {mon(fa)}. ⭐ The rating dropped visibly in {mon(notice)}, "
               f"<b>{months_between(fa, notice)} months later</b>. That is the head start.")
    return fig, f'<div class="banner">{msg}</div>'

def lead_table_html(R):
    rows = [r for r in R["rows"] if r["first_alert"] is not None]
    if not rows: return '<div class="banner">No alerts fired at this sensitivity.</div>'
    body = "".join(f'<tr><td><b>{r["aspect"]}</b></td><td>{mon(r["first_alert"])}</td><td>{mon(R["notice"])}</td>'
                   f'<td><b>{"—" if r["lead"] is None else (str(r["lead"]) + " months" if r["lead"] > 0 else "none: rating had already dropped")}</b></td></tr>' for r in sorted(rows, key=lambda r: r["first_alert"]))
    return f'<table><tr><th>Dish or moment</th><th>DishRadar alert</th><th>Overall rating visibly drops</th><th>Head start</th></tr>{body}</table>'
print("Components ready ✔")

Components ready ✔


## 5 · Results (printed for your report)

In [ ]:
print("=" * 60)
print(f"Reviews: {len(reviews):,} | extracted mentions: {len(mentions):,} | polarity accuracy vs ground truth: {acc:.1%}")
print(f"Overall rating baseline {R['rbase']:.2f}★; visible drop in {mon(R['notice'])}")
for r in sorted([r for r in R["rows"] if r["first_alert"] is not None], key=lambda r: r["first_alert"]):
    print(f"  🔴 {r['aspect']:<22} alert {mon(r['first_alert'])}  | head start: {r['lead']} months" + ("  (rating had already dropped)" if r['lead'] is not None and r['lead'] <= 0 else ""))
print("Planted truth: biryani decline began Oct 2025; service speed decline began Feb 2026.")
fa_names = {r["aspect"] for r in R["rows"] if r["first_alert"] is not None}
false_pos = fa_names - {"Chicken Dum Biryani", "Service speed"}
print("False alarms:", ", ".join(false_pos) if false_pos else "none")
print("Not enough data (guardrail):", ", ".join(r["aspect"] for r in R["rows"] if r["status"] == "Not enough data") or "none")
print("=" * 60)
tbl.to_csv("dishradar_results.csv", index=False); mentions.to_csv("dishradar_mentions.csv", index=False)
print("Saved dishradar_results.csv and dishradar_mentions.csv (see the Files panel on the left)")
# Limitations to mention: synthetic data; keyword + lexicon sentiment misses sarcasm; sentiment is per clause, not per sentence pair; one restaurant.

Reviews: 1,193 | extracted mentions: 2,717 | polarity accuracy vs ground truth: 98.9%
Overall rating baseline 4.31★; visible drop in Mar 2026
  🔴 Chicken Dum Biryani    alert Jan 2026  | head start: 2 months
  🔴 Service speed          alert May 2026  | head start: -2 months  (rating had already dropped)
Planted truth: biryani decline began Oct 2025; service speed decline began Feb 2026.
False alarms: none
Not enough data (guardrail): Ambience, Veg Manchurian
Saved dishradar_results.csv and dishradar_mentions.csv (see the Files panel on the left)


## 6 · The dashboard
Five tabs, each opening with a short "how to read this" guide. For the live demo, open **Time machine**, pick the biryani, press ▶ and narrate.

In [ ]:
z_sl = w.FloatSlider(value=1.5, min=.8, max=3, step=.1, description="Sensitivity", continuous_update=False, readout_format=".1f", style={"description_width": "90px"}, layout=w.Layout(width="320px"))
n_sl = w.IntSlider(value=6, min=3, max=15, description="Min mentions", continuous_update=False, style={"description_width": "90px"}, layout=w.Layout(width="320px"))
d_sl = w.FloatSlider(value=.3, min=.1, max=.5, step=.05, description="Star drop noticed", continuous_update=False, readout_format=".2f", style={"description_width": "110px"}, layout=w.Layout(width="340px"))
tip = w.HTML(f'{CSS}<div class="dr muted" style="padding:8px 12px">Sensitivity: lower alerts sooner but risks false alarms. Min mentions: months with fewer mentions are ignored. Star drop: how far the 3-month rating must fall before a manager would notice.</div>')

out_radar, out_deep, out_proof, out_try, out_data = (w.Output() for _ in range(5))
dd_deep = w.Dropdown(description="Dish", layout=w.Layout(width="320px"))
dd_proof = w.Dropdown(description="Dish", layout=w.Layout(width="320px"))
t_sl = w.IntSlider(value=0, min=0, max=1, description="Month", continuous_update=False, readout=False, layout=w.Layout(width="480px"))
t_lbl = w.HTML()
play = w.Play(interval=900, min=0, max=1, step=1, value=0, description="Replay")
w.jslink((play, "value"), (t_sl, "value")); w.jslink((play, "min"), (t_sl, "min")); w.jslink((play, "max"), (t_sl, "max"))
S = {}

def render_deep(*_):
    with out_deep:
        clear_output(wait=True)
        if not dd_deep.value: return
        a = dd_deep.value; display(wrap(guide("deep"))); display(fig_deep(S["R"], a))
        display(wrap(f'<div class="banner">{badge(S["R"]["rows"][[r["aspect"] for r in S["R"]["rows"]].index(a)]["status"])} &nbsp; {a}</div>' + complaint_html(a, S["R"])))

def render_proof(*_):
    with out_proof:
        clear_output(wait=True)
        if not dd_proof.value: return
        R_ = S["R"]; i = t_sl.value; t_lbl.value = f'<b style="font-size:15px">{mon(R_["score"].index[min(i, len(R_["score"]) - 1)])}</b>'
        fig, msg = fig_proof(R_, dd_proof.value, i)
        display(wrap(guide("time") + msg)); display(fig); display(wrap("<h2 style='font-size:19px;margin:4px 0 8px'>Head start for every alert</h2>" + lead_table_html(R_)))

def render_radar():
    with out_radar:
        clear_output(wait=True); display(wrap(radar_html(S["R"]))); display(fig_heat(S["R"]))

def recompute(*_):
    S["R"] = analyse(S["m"], S["df"], z_sl.value, n_sl.value, d_sl.value)
    names = [r["aspect"] for r in S["R"]["rows"] if r["status"] != "Not enough data"]
    best = max([r for r in S["R"]["rows"] if r["lead"] is not None] or S["R"]["rows"], key=lambda r: (r["lead"] or -99) if r["first_alert"] is not None else -99)["aspect"]
    for dd in (dd_deep, dd_proof):
        cur = dd.value
        target = cur if cur in names else (best if best in names else (names[0] if names else None))
        for fn in (render_deep, render_proof):
            try: dd.unobserve(fn, "value")
            except ValueError: pass
        dd.options = names
        dd.value = target
    dd_deep.observe(render_deep, "value"); dd_proof.observe(render_proof, "value")
    t_sl.max = play.max = len(S["R"]["score"]) - 1
    if t_sl.value == 0: t_sl.value = t_sl.max
    render_radar(); render_deep(); render_proof()

def load(df, engine="lexicon"):
    df = df.copy(); df["month"] = df.date.dt.to_period("M").dt.to_timestamp()
    S["df"] = df; S["m"] = build_mentions(df, engine)
    global reviews; reviews = df

for s_ in (z_sl, n_sl, d_sl): s_.observe(recompute, "value")
t_sl.observe(render_proof, "value")

# -- Try a review
ta = w.Textarea(value="The biryani was dry and bland today, but the butter chicken was delicious. Service was painfully slow. Lovely ambience though.",
                layout=w.Layout(width="95%", height="90px"))
btn = w.Button(description="Read this review", button_style="primary", icon="search")
def on_try(_):
    with out_try:
        clear_output(wait=True)
        items = extract_lexicon(ta.value); overall = sia.polarity_scores(ta.value)["compound"]
        chips = "".join(f'<div class="quote {"pos" if s_ > .05 else ""}"><b>{a}</b> · {s_:+.2f}<br><span class="muted">“{c}”</span></div>' for a, k, s_, c in items) or '<div class="muted">No dish or service moment found. Check the menu words in Section 2.</div>'
        display(wrap(f'<div class="banner">A normal overall score says <b>{overall:+.2f}</b>, which hides the detail. DishRadar splits it into:</div>{chips}'))
btn.on_click(on_try)

# -- Use your data
up = w.FileUpload(accept=".csv", multiple=False, description="Upload CSV")
menu_in = w.Text(placeholder="e.g. Hyderabadi biryani, Chicken 65, Mutton Haleem", description="Your dishes", layout=w.Layout(width="95%"), style={"description_width": "90px"})
btn2 = w.Button(description="Analyse my reviews", button_style="success", icon="play")
def on_data(_):
    with out_data:
        clear_output(wait=True)
        if not up.value: print("Upload a CSV first (columns: date, text, rating)."); return
        f = up.value[0] if isinstance(up.value, (list, tuple)) else list(up.value.values())[0]
        raw = pd.read_csv(io.BytesIO(bytes(f["content"])))
        low = {c.lower(): c for c in raw.columns}
        pick = lambda *k: next((low[x] for x in k if x in low), None)
        dc, tc, rc = pick("date", "time", "review_date"), pick("text", "review", "content", "review_text"), pick("rating", "stars", "score")
        if not (dc and tc and rc): print("Need columns for date, review text and rating. Found:", list(raw.columns)); return
        df = pd.DataFrame({"review_id": range(len(raw)), "date": pd.to_datetime(raw[dc], errors="coerce"), "rating": pd.to_numeric(raw[rc], errors="coerce"), "text": raw[tc].astype(str)}).dropna(subset=["date", "rating"])
        names = [x.strip() for x in menu_in.value.split(",") if x.strip()]
        if not names: print("Type at least one dish name in 'Your dishes'."); return
        set_menu({n: [n.lower()] for n in names})
        print(f"Analysing {len(df):,} reviews…"); load(df); recompute()
        print("Done. The other tabs now show your data.")
btn2.on_click(on_data)

load(reviews)
tabs = w.Tab(children=[out_radar,
                       w.VBox([dd_deep, out_deep]),
                       w.VBox([w.HBox([dd_proof, play, t_sl, t_lbl]), out_proof]),
                       w.VBox([w.HTML(CSS + '<div class="dr">' + guide("try") + '</div>'), ta, btn, out_try]),
                       w.VBox([w.HTML(CSS + '<div class="dr">' + guide("data") + '</div>'), up, menu_in, btn2, out_data])])
for i, t in enumerate(["🚨 Radar", "🔍 Dish deep-dive", "⏱ Time machine", "🧪 Try a review", "📂 Use your data"]): tabs.set_title(i, t)
display(w.VBox([w.HBox([z_sl, n_sl, d_sl]), tip, tabs]))
recompute()

## 7 · Turn the notebook into a website
This cell packs the notebook's results into one self-contained web page, `dishradar_site.html`, with the same Radar, Deep-dive and Time machine views. It runs in any browser, so you can share it without sharing the notebook.

* **Instant public link:** the next cell starts a Gradio server and prints a `https://….gradio.live` URL. It stays live for about a week while this notebook keeps running.
* **Permanent link:** download `dishradar_site.html` and drag it onto [Netlify Drop](https://app.netlify.com/drop) or upload it to a GitHub Pages repo.

In [ ]:
import json, html
SITE_TEMPLATE = r'''<!DOCTYPE html>
<html lang="en"><head><meta charset="utf-8">
<meta name="viewport" content="width=device-width, initial-scale=1, viewport-fit=cover">
<title>DishRadar · find the dish that's dragging your rating</title>
<link href="https://fonts.googleapis.com/css2?family=Fraunces:opsz,wght@9..144,600&family=Hanken+Grotesk:wght@400;600;700&display=swap" rel="stylesheet">
<script src="https://cdnjs.cloudflare.com/ajax/libs/Chart.js/4.4.1/chart.umd.min.js"></script>
<style>
:root{--bg:#EEF4F3;--card:#fff;--ink:#0E2229;--mut:#506A6E;--line:#CFE0DE;--good:#0E8F63;--warn:#A86F00;--bad:#D63A2F;--info:#1B7FA6;--acc:#F4B223;--hero:#15303A;box-sizing:border-box;padding-top:env(safe-area-inset-top,0px);padding-bottom:env(safe-area-inset-bottom,0px)}
@media (prefers-color-scheme:dark){:root:not([data-theme="light"]){--bg:#0E2229;--card:#15303A;--ink:#EAF2F0;--mut:#8FB0B3;--line:#26474F;--good:#5FD6A0;--warn:#F4B223;--bad:#FF5A4E;--info:#6EC1E4;--hero:#1d4350}}
:root[data-theme="dark"]{--bg:#0E2229;--card:#15303A;--ink:#EAF2F0;--mut:#8FB0B3;--line:#26474F;--good:#5FD6A0;--warn:#F4B223;--bad:#FF5A4E;--info:#6EC1E4;--hero:#1d4350}
html{scroll-padding-top:env(safe-area-inset-top,0px)}
*{box-sizing:border-box}body{margin:0;background:var(--bg);color:var(--ink);font:15px/1.55 'Hanken Grotesk',system-ui,sans-serif}
.wrap{max-width:1040px;margin:0 auto;padding:16px}
h1,h2{font-family:Fraunces,Georgia,serif;margin:0}h2{font-size:20px;margin:18px 0 8px}
.hero{background:linear-gradient(120deg,#15303A,var(--hero));color:#EAF2F0;border-radius:16px;padding:22px 24px;display:flex;gap:16px;justify-content:space-between;align-items:flex-start}
.hero h1{font-size:30px}.hero p{margin:6px 0 0;color:#B9D3D4;max-width:640px}
button,select,input,textarea{font:inherit;color:inherit}
.btn{background:var(--acc);color:#0E2229;border:0;border-radius:8px;padding:8px 14px;font-weight:700;cursor:pointer}
.ghost{background:transparent;color:#EAF2F0;border:1px solid #5b7f86}
:focus-visible{outline:3px solid var(--info);outline-offset:2px}
.ctrl{display:flex;flex-wrap:wrap;gap:8px 22px;margin:14px 0;padding:12px 16px;background:var(--card);border:1px solid var(--line);border-radius:12px}
.ctrl label{display:flex;flex-direction:column;font-size:13px;color:var(--mut);min-width:200px}.ctrl b{color:var(--ink)}
.ctrl small{flex-basis:100%;color:var(--mut)}
nav{display:flex;gap:6px;overflow-x:auto;border-bottom:1px solid var(--line)}
nav button{background:none;border:0;border-bottom:3px solid transparent;padding:10px 14px;font-weight:600;color:var(--mut);cursor:pointer;white-space:nowrap}
nav button[aria-selected=true]{color:var(--ink);border-color:var(--acc)}
.guide{background:var(--card);border:1px solid var(--line);border-left:5px solid var(--acc);border-radius:10px;padding:12px 16px;margin:14px 0;display:grid;gap:4px}
.guide span{color:var(--mut)}.guide b{color:var(--ink)}
.kpis{display:grid;grid-template-columns:repeat(auto-fit,minmax(170px,1fr));gap:10px}
.kpi{background:var(--card);border:1px solid var(--line);border-radius:10px;padding:10px 14px}.kpi small{color:var(--mut)}.kpi b{display:block;font-size:24px}
.card{background:var(--card);border:1px solid var(--line);border-radius:10px;padding:12px 16px;margin:10px 0}
.card.bad{border-left:5px solid var(--bad)}.card.good{border-left:5px solid var(--good)}
.tw{overflow-x:auto}table{width:100%;border-collapse:collapse;background:var(--card);border-radius:10px;font-size:14px}
th{text-align:left;color:var(--mut);padding:8px 10px;font-size:13px}td{padding:8px 10px;border-top:1px solid var(--line)}
.bd{padding:2px 10px;border-radius:99px;font-weight:700;font-size:12px;border:1px solid currentColor;white-space:nowrap}
.Alert{color:var(--bad)}.Watch{color:var(--warn)}.Healthy{color:var(--good)}.Improving{color:var(--info)}.Not{color:var(--mut)}
.chip{display:inline-block;margin:3px 6px 3px 0;padding:3px 10px;border-radius:99px;background:var(--bg);border:1px solid var(--line);font-size:13px}
.q{border-left:4px solid var(--bad);background:var(--card);padding:6px 12px;margin:6px 0;border-radius:6px}.q.p{border-color:var(--good)}
.cv{position:relative;height:340px;background:var(--card);border:1px solid var(--line);border-radius:10px;padding:8px}
.cv.s{height:170px}
.row{display:flex;flex-wrap:wrap;gap:12px;align-items:center;margin:8px 0}
select,textarea,input[type=range]{background:var(--card);border:1px solid var(--line);border-radius:8px;padding:6px 10px}
input[type=range]{padding:0;accent-color:var(--acc)}textarea{width:100%;min-height:90px}
.hm td{padding:0;border:1px solid var(--bg);text-align:center;font-size:11px;color:#fff;height:26px;min-width:34px}.hm td.n{text-align:left;padding:0 8px;color:var(--ink);font-size:13px;white-space:nowrap;background:var(--card)}
.mut{color:var(--mut)}.ai{border-left:5px solid var(--info)}.pre{white-space:pre-wrap}.drop{border:2px dashed var(--line);border-radius:12px;padding:18px;text-align:center;background:var(--card)}.drop.on{border-color:var(--acc)}.in{width:100%;padding:8px 10px;background:var(--card);border:1px solid var(--line);border-radius:8px}code{background:var(--bg);padding:1px 6px;border-radius:6px}[hidden]{display:none!important}
@media(max-width:640px){.hero{flex-direction:column}}
</style></head><body><div class="wrap">
<header class="hero"><div><h1>DishRadar</h1><p id="sub"></p></div><button class="btn ghost" id="th">Switch theme</button></header>
<div class="ctrl" aria-label="Settings">
<label>Sensitivity: <b id="zv"></b><input id="z" type="range" min="0.8" max="3" step="0.1" value="1.5"></label>
<label>Minimum mentions per month: <b id="nv"></b><input id="mn" type="range" min="3" max="15" step="1" value="6"></label>
<label>Rating drop a manager would notice: <b id="dv"></b>★<input id="dr" type="range" min="0.1" max="0.5" step="0.05" value="0.3"></label>
<small>Lower sensitivity warns sooner but may raise false alarms. Months with fewer mentions than the minimum are ignored. Every tab updates when you move these.</small></div>
<nav role="tablist" id="tabs"></nav>

<section id="p0" role="tabpanel">
<div class="guide"><b>Radar: which dishes need attention today?</b><span><b>Read it:</b> each row is a dish or service moment. The status compares its last 3 months with its own first 6 months. The line shows its sentiment over time. In the heatmap, red means guests are unhappy and green means happy; grey means too few mentions.</span><span><b>Do this:</b> start with the red rows, then read the suggested next step.</span><span><b>Words used:</b> <b>Dish</b> = something on the menu. <b>Moment</b> = part of the visit such as service speed or ambience. <b>Baseline</b> = how happy guests normally were in the first 6 months (scale: -1 angry to +1 delighted). <b>Last 3 mo</b> = the same score for the latest 3 months. <b>Times mentioned</b> = how many times guests talked about this dish or moment in all reviews, one count per sentence about it. More mentions means the score is more trustworthy.</span></div>
<div class="kpis" id="kp"></div><div id="act"></div><h2>🍛 Dishes</h2><div class="tw"><table id="rt"></table></div><h2>🛎 Service and atmosphere moments</h2><div class="tw"><table id="rt2"></table></div>
<h2>Heatmap: dishes by month</h2><div class="tw"><table class="hm" id="hm"></table></div><h2>Heatmap: moments by month</h2><div class="tw"><table class="hm" id="hm2"></table></div></section>

<section id="p1" role="tabpanel" hidden>
<div class="guide"><b>Deep-dive: why is this one flagged?</b><span><b>Read it:</b> the blue line is how happy guests are each month (higher is happier). If it falls below the red dotted line, DishRadar raises an alert.</span><span><b>Do this:</b> read the summary, then the complaints, to find what changed.</span></div>
<div class="row"><label>Pick a dish or moment <select id="dd1"></select></label><span id="st1"></span></div>
<div id="sm"></div><div class="cv"><canvas id="c1"></canvas></div><div id="cm"></div></section>

<section id="p2" role="tabpanel" hidden>
<div class="guide"><b>Time machine: would this really have warned us early?</b><span><b>Read it:</b> drag the month or press Play. You only see what a manager would have known at that point. The blue line is the dish, the amber line is the overall rating.</span><span><b>New:</b> the AI predictive model below uses only the months up to the one you picked to forecast the next 3 months, with a likely range, the chance of an alert, and a backtest showing how accurate it has been.</span><span><b>Look for:</b> the red line (dish alert) appearing before the amber line (rating visibly drops). The gap is the head start.</span></div>
<div class="row"><label>Dish <select id="dd2"></select></label><button class="btn" id="pl">▶ Play</button><input id="tm" type="range" min="0" value="0" style="flex:1;min-width:200px"><b id="tl"></b></div>
<div id="bn"></div><div class="cv"><canvas id="c2"></canvas></div><h2>🤖 AI predictive model: likely ratings for this dish</h2><div id="pm"></div><div class="cv"><canvas id="c2p"></canvas></div><div class="row"><button class="btn" id="aib">🤖 Explain this forecast with AI</button><span class="mut" id="ais"></span></div><div id="aiout"></div><h2>Head start for every alert</h2><div class="tw" id="lt"></div></section>

<section id="p3" role="tabpanel" hidden>
<div class="guide"><b>Try a review: see what a single star rating hides.</b><span><b>Read it:</b> paste a review. The overall score blends everything, then DishRadar splits it by dish. This is a simplified version of the notebook's scorer.</span><span><b>Do this:</b> write one review with a good dish and a bad one, then compare.</span></div>
<textarea id="tx">The biryani was dry and bland today, but the butter chicken was delicious. Service was painfully slow. Lovely ambience though.</textarea>
<div class="row"><button class="btn" id="go">Read this review</button></div><div id="tr"></div></section>

<section id="p4" role="tabpanel" hidden>
<div class="guide"><b>Use your data: run DishRadar on any restaurant</b><span><b>Read it:</b> upload a CSV of reviews, list the dishes to track, and every tab (Radar, Deep-dive, Time machine, Try a review) switches to your restaurant.</span><span><b>Privacy:</b> files are read in your browser and are not uploaded anywhere. Only when you press an AI button, summary numbers (not your reviews) are sent to Claude.</span></div>
<div class="card"><b>1. Restaurant name</b><br><input class="in" id="un" placeholder="e.g. Green Leaf Bistro"></div>
<div class="card"><b>2. Upload reviews (CSV)</b><div class="drop" id="dz"><p>Drop a .csv file here or</p><input type="file" id="uf" accept=".csv,text/csv"><p class="mut" id="ufn">No file chosen</p></div>
<p class="mut">Needs a header row with a <b>date</b> column, a <b>review</b> text column and a <b>rating</b> (1 to 5) column. Names like <code>date</code>, <code>review</code>/<code>text</code>/<code>comment</code>, <code>rating</code>/<code>stars</code> are found automatically. At least 9 months of history is needed. Several restaurants? Run them one at a time and switch by uploading the next file.</p><button class="btn" id="tpl" hidden>Download a template CSV</button></div>
<div class="card"><b>3. Dishes to track</b> (comma separated)<textarea id="ud" placeholder="Margherita pizza, Pad thai, Tiramisu"></textarea><div class="row"><button class="btn" id="sg">🤖 Suggest dishes from my reviews</button><span class="mut" id="sgs"></span></div><span class="mut">Service speed, Staff behaviour, Ambience and Value for money are always tracked.</span></div>
<div class="row"><button class="btn" id="run">Analyse my data</button><button class="btn ghost" style="color:var(--ink);border-color:var(--line)" id="rs">Back to demo restaurant</button></div>
<div id="um"></div>
<div class="card bad"><b>Honest limits.</b> The scorer is word-based and can miss sarcasm. Dish names must appear the way you type them. Forecasts are statistical estimates, not guarantees. The demo restaurant uses <span id="dn"></span> simulated reviews.</div></section>
</div>
<script>
let D=/*DATA*/null;
Chart.defaults.font.family="'Hanken Grotesk',system-ui,sans-serif";Chart.defaults.maintainAspectRatio=false;
const $=s=>document.querySelector(s);let M=D.months,B=6,K=3,ST={Alert:'🔴',Watch:'🟡',Healthy:'🟢',Improving:'🔵','Not enough data':'⚪'};
const mean=a=>a.reduce((x,y)=>x+y,0)/a.length,sd=a=>{const m=mean(a);return Math.sqrt(a.reduce((s,x)=>s+(x-m)**2,0)/(a.length-1))};
const cs=n=>getComputedStyle(document.documentElement).getPropertyValue('--'+n).trim(),cl=v=>v.filter(x=>x!==null);
const bd=s=>`<span class="bd ${s.split(' ')[0]}">${ST[s]} ${s}</span>`,f2=v=>v==null||isNaN(v)?'':v.toFixed(2);
function assess(s,z){const base=cl(s.slice(0,B)),rec=cl(s.slice(-K));
 if(s.length<B+K||base.length<3||rec.length<2)return{st:'Not enough data'};
 const mu=mean(base),sg=Math.max(sd(base),.12),pts=[];s.slice(-6).forEach((v,i)=>{if(v!==null)pts.push([i,v])});
 let sl=0;if(pts.length>=3){const mx=mean(pts.map(p=>p[0])),my=mean(pts.map(p=>p[1]));sl=pts.reduce((a,p)=>a+(p[0]-mx)*(p[1]-my),0)/pts.reduce((a,p)=>a+(p[0]-mx)**2,0)}
 const rc=mean(rec),dp=mu-rc,zs=dp/sg;let st='Healthy';
 if(zs>=z)st='Alert';else if(zs>=z*.5||(sl<-.04&&zs>0))st='Watch';else if(zs<=-z)st='Improving';
 return{st,mu,sg,rc,dp,zs}}
function firstAlert(s,z){for(let t=B+K;t<=s.length;t++)if(assess(s.slice(0,t),z).st==='Alert')return t-1;return null}
function notice(stars,dr){const roll=stars.map((_,i)=>i<2?null:mean(stars.slice(i-2,i+1))),base=mean(stars.slice(0,B));
 for(let i=B;i<roll.length;i++)if(roll[i]!==null&&roll[i]<=base-dr)return{i,roll,base};return{i:null,roll,base}}
let z=1.5,mn=6,dr=.3,R=[],RN,tmI=M.length-1,ch={},timer=null;
function compute(){RN=notice(D.stars,dr);R=D.aspects.map(a=>{const s=a.score.map((v,i)=>a.n[i]>=mn?v:null),r=assess(s,z),fa=firstAlert(s,z);
 return{...a,s,r,fa,lead:fa!==null&&RN.i!==null?RN.i-fa:null,tot:a.n.reduce((x,y)=>x+y,0)}})}
const ord={Alert:0,Watch:1,Healthy:2,Improving:3,'Not enough data':4};
const spark=(s,c)=>{const p=s.map((v,i)=>[i,v]).filter(x=>x[1]!==null);if(p.length<2)return'';const lo=Math.min(...p.map(x=>x[1])),hi=Math.max(...p.map(x=>x[1])),sp=hi-lo||1;
 return`<svg width="110" height="26" aria-hidden="true"><polyline fill="none" stroke="${c}" stroke-width="2" points="${p.map(x=>(x[0]/(s.length-1)*110).toFixed(1)+','+(24-(x[1]-lo)/sp*22).toFixed(1)).join(' ')}"/></svg>`};
const colr=st=>cs({Alert:'bad',Watch:'warn',Healthy:'good',Improving:'info'}[st]||'mut');
function mk(id,cfg){try{ch[id]&&ch[id].destroy()}catch(e){}ch[id]=null;let cv=$('#'+id);const n=cv.cloneNode(false);cv.replaceWith(n);Chart.defaults.color=cs('mut');try{ch[id]=new Chart(n,cfg)}catch(e){console.error(e)}}
Chart.register({id:'vl',afterDatasetsDraw(c){const L=c.options.plugins.vl||[],{ctx,chartArea:a,scales:{x}}=c;L.forEach(l=>{const px=x.getPixelForValue(l.i),r=px>a.right-190;ctx.save();ctx.strokeStyle=l.c;ctx.fillStyle=l.c;ctx.lineWidth=2;ctx.setLineDash([5,4]);ctx.beginPath();ctx.moveTo(px,a.top);ctx.lineTo(px,a.bottom);ctx.stroke();ctx.setLineDash([]);ctx.font='700 12px sans-serif';ctx.textAlign=r?'right':'left';ctx.fillText(l.t,px+(r?-6:6),a.top+(l.y||14));ctx.restore()})}});
const act=(a)=>{const c=D.complaints[a.name]||{r:{},e:{}},top=Object.entries(c.r).sort((x,y)=>y[1]-x[1]).slice(0,3);
 const w=top.map(([k,v])=>`“${k}” in ${v} recent complaints (only ${c.e[k]||0} in the first 6 months)`).join('; ');
 const fix=a.kind==='Dish'?'Check the recipe, the ingredient supplier and who is cooking it on busy shifts.':'Check staffing and rota at peak hours.';
 return`<div class="card bad"><b>Suggested next step for ${a.name}</b><br>${fix}${w?' In the last 3 months guests complain about: '+w+'.':''}</div>`};
function radar(){const al=R.filter(r=>r.r.st==='Alert').sort((a,b)=>b.r.zs-a.r.zs),rows=[...R].sort((a,b)=>ord[a.r.st]-ord[b.r.st]||(b.r.zs||-9)-(a.r.zs||-9)),
 now=cl(RN.roll).slice(-1)[0],ld=al.map(r=>r.lead).filter(x=>x!==null&&x>0);
 $('#kp').innerHTML=[['Reviews analysed',D.nReviews.toLocaleString()],['Active alerts',`<span style="color:${al.length?cs('bad'):cs('good')}">${al.length}</span>`],['Biggest head start',ld.length?Math.max(...ld)+' months':'—'],['Overall rating now',now.toFixed(2)+'★ <small>vs '+RN.base.toFixed(2)+'★ before</small>']].map(k=>`<div class="kpi"><small>${k[0]}</small><b>${k[1]}</b></div>`).join('');
 $('#act').innerHTML=al.length?`<div class="card bad">🔴 <b>${al[0].name}</b> has been below its healthy range since <b>${M[al[0].fa]}</b>${al[0].lead>0?`, <b>${al[0].lead} months</b> before the overall rating visibly dropped`:''}.</div>`+act(al[0]):'<div class="card good">🟢 Nothing is below its healthy range at these settings.</div>';

 const hdr=l=>`<tr><th>${l}</th><th>Status</th><th>Trend</th><th>Baseline</th><th>Last 3 mo</th><th>Alert since</th><th title="How many times guests talked about it">Times mentioned</th></tr>`,
 line=r=>`<tr><td><b>${r.name}</b></td><td>${bd(r.r.st)}</td><td>${spark(r.s,colr(r.r.st))}</td><td>${f2(r.r.mu)}</td><td>${f2(r.r.rc)}</td><td>${r.fa!==null?M[r.fa]:'—'}</td><td>${r.tot}</td></tr>`,
 tbl=(id,kind,l)=>{const x=rows.filter(r=>r.kind===kind);$(id).innerHTML=x.length?hdr(l)+x.map(line).join(''):'<tr><td class="mut">None found in this data.</td></tr>'},
 hmap=(id,kind)=>{const x=rows.filter(r=>r.kind===kind&&r.r.st!=='Not enough data');$(id).innerHTML=x.length?'<tr><td class="n"></td>'+M.map(m=>`<td class="n" style="font-size:10px;text-align:center;padding:0">${m.slice(0,3)}<br>${m.slice(-2)}</td>`).join('')+'</tr>'+x.map(r=>`<tr><td class="n">${r.name}</td>${r.s.map(v=>v===null?'<td style="background:#8887"></td>':`<td title="${v.toFixed(2)}" style="background:hsl(${Math.max(0,Math.min(1,(v+.1)/.8))*125} 55% 42%)">${v.toFixed(1)}</td>`).join('')}</tr>`).join(''):'<tr><td class="mut">Nothing to show.</td></tr>'};
 tbl('#rt','Dish','Dish');tbl('#rt2','Moment','Moment');hmap('#hm','Dish');hmap('#hm2','Moment')}
function ewma(s){let p=null;return s.map(v=>{if(v===null)return p;p=p===null?v:.5*v+.5*p;return p})}
function deep(){const a=R.find(r=>r.name===$('#dd1').value);if(!a)return;$('#st1').innerHTML=bd(a.r.st);const r=a.r,f=v=>v.toFixed(2);
 const T={Alert:`Guests are clearly less happy than before. Their score fell from <b>${f(r.mu)}</b> to <b>${f(r.rc)}</b>.`,Watch:`Guests are a little less happy than before (<b>${f(r.mu)}</b> to <b>${f(r.rc)}</b>). Keep an eye on it.`,Healthy:`Nothing unusual. Guests are about as happy as before (<b>${f(r.mu)}</b> then, <b>${f(r.rc)}</b> now).`,Improving:`Guests are happier than before (<b>${f(r.mu)}</b> to <b>${f(r.rc)}</b>).`};
 $('#sm').innerHTML=`<div class="card ${r.st==='Alert'?'bad':r.st==='Alert'||r.st==='Watch'?'':'good'}">${T[r.st]||''}${a.fa!==null?` The alert first fired in <b>${M[a.fa]}</b>.`:''} <span class="mut">(Score runs from -1 angry to +1 delighted. Mentioned ${a.tot} times in total.)</span></div>`;
 const d=[{label:'How happy guests are',data:a.s,borderColor:cs('info'),backgroundColor:cs('info'),borderWidth:3,pointRadius:4}];
 if(r.mu!==undefined)d.push({label:'Alert line',data:M.map(_=>r.mu-z*r.sg),borderColor:cs('bad'),borderDash:[4,4],pointRadius:0,borderWidth:2});
 mk('c1',{type:'line',data:{labels:M,datasets:d},options:{animation:false,plugins:{title:{display:true,text:a.name,color:cs('ink')}},scales:{y:{title:{display:true,text:'Happiness score'},grid:{color:cs('line')}},x:{grid:{color:cs('line')}}}}});
 const c=D.complaints[a.name]||{r:{},e:{},rq:[],eq:[]},ch_=Object.entries(c.r).sort((x,y)=>y[1]-x[1]).map(([k,v])=>`<span class="chip"><b>${k}</b>: ${v} complaints in the last 3 months <span class="mut">(${c.e[k]||0} in the first 6 months)</span></span>`).join('')||'<span class="mut">No complaint words in the last 3 months.</span>';
 $('#cm').innerHTML=`<h2>What guests complain about now</h2><div>${ch_}</div><h2>Recent complaints</h2>${c.rq.map(q=>`<div class="q">“${q}”</div>`).join('')||'<span class="mut">None.</span>'}<h2>How they talked about it early on</h2>${c.eq.map(q=>`<div class="q p">“${q}”</div>`).join('')||'<span class="mut">None.</span>'}`}
function tm(){const a=R.find(r=>r.name===$('#dd2').value);if(!a)return;const i=tmI,s=a.s.slice(0,i+1),st=D.stars.slice(0,i+1),n=notice(st,dr),fa=firstAlert(s,z),r=assess(s,z),cur=cl(n.roll).slice(-1)[0];
 $('#tl').textContent=M[i];let m;
 if(i+1<B+K)m='Learning the baseline from the first six months of reviews.';
 else if(fa===null)m=`${ST[r.st]} <b>${a.name}</b> status: ${r.st}. Overall rating: ${cur.toFixed(2)}★.`;
 else if(n.i===null)m=`🔴 <b>Alert fired in ${M[fa]}.</b> The overall rating is still <b>${cur.toFixed(2)}★</b> and looks fine. Without DishRadar nobody would be looking at ${a.name} yet.`;
 else m=`🔴 Alert fired in ${M[fa]}. ⭐ The rating visibly dropped in ${M[n.i]}, <b>${n.i-fa} months ${n.i>=fa?'later':'earlier'}</b>.${n.i-fa>0?' That is the head start.':''}`;
 $('#bn').innerHTML=`<div class="card ${fa!==null?'bad':'good'}">${m}</div>`;
 const L=[];if(fa!==null)L.push({i:fa,c:cs('bad'),t:'🔴 dish alert'});if(n.i!==null)L.push({i:n.i,c:cs('warn'),t:'⭐ rating drops',y:32});
 const all=cl(a.s.map((v,j)=>v)),roll=notice(D.stars,0).roll;
 mk('c2',{type:'line',data:{labels:M,datasets:[{label:a.name+' sentiment (left)',data:a.s.map((v,j)=>j<=i?v:null),borderColor:cs('info'),backgroundColor:cs('info'),borderWidth:3,pointRadius:4,yAxisID:'y'},{label:'Overall rating, 3-month average (right)',data:n.roll.concat(Array(M.length-n.roll.length).fill(null)),borderColor:cs('acc'),backgroundColor:cs('acc'),borderWidth:3,pointRadius:3,yAxisID:'y1'}]},
 options:{animation:false,plugins:{vl:L},scales:{y:{min:Math.min(...all)-.1,max:Math.max(...all)+.1,title:{display:true,text:'Dish sentiment'},grid:{color:cs('line')}},y1:{position:'right',min:Math.min(...cl(roll))-.15,max:Math.max(...cl(roll))+.15,title:{display:true,text:'Stars'},grid:{display:false}},x:{grid:{color:cs('line')}}}}});
 const ar=R.filter(r=>r.fa!==null).sort((a,b)=>a.fa-b.fa);
 fc(a,i,r);$('#lt').innerHTML=ar.length?'<table><tr><th>Dish or moment</th><th>DishRadar alert</th><th>Rating visibly drops</th><th>Head start</th></tr>'+ar.map(r=>`<tr><td><b>${r.name}</b></td><td>${M[r.fa]}</td><td>${RN.i!==null?M[RN.i]:'—'}</td><td><b>${r.lead===null?'—':r.lead>0?r.lead+' months':'none: rating had already dropped'}</b></td></tr>`).join('')+'</table>':'<div class="card">No alerts at these settings.</div>'}
const PW={delicious:2.5,tasty:2,amazing:3,excellent:3,good:1.9,great:3,lovely:2.8,loved:3,enjoyed:2,perfect:2.7,perfectly:2.2,fresh:1.5,friendly:2.2,helpful:1.8,polite:1.3,warm:1.3,comfortable:1.5,pleasant:1.8,rich:1.2,generous:2,worth:1,quick:1.5,prompt:1.5,promptly:1.5,attentive:1.8,flavourful:2,seasoned:1.2,authentic:1.8,juicy:1.8,aromatic:2},
NW={dry:-1.8,bland:-2,cold:-1.3,stale:-2,overcooked:-2,undercooked:-2,greasy:-1.8,tasteless:-2.4,soggy:-1.8,rubbery:-1.8,terrible:-2.6,slow:-1.6,rude:-2.4,noisy:-1.5,cramped:-1.8,overpriced:-2.2,expensive:-1.2,tiny:-1,disappointed:-2.2,disappointing:-2.2,bad:-2.5,awful:-3,worst:-3.1,ignored:-1.5,unhelpful:-1.7,dull:-1.7,uncomfortable:-1.6,hardly:-.9,painfully:-1,inattentive:-2,forever:-1,waited:-.8,mediocre:-1.5,letdown:-1.5};
function sc(c){const t=c.toLowerCase().match(/[a-z']+/g)||[];let s=0;t.forEach((w,i)=>{let v=PW[w]||NW[w]||0;if(v&&t.slice(Math.max(0,i-2),i).some(x=>x==='not'||x==='never'||x==='no'||x.endsWith("n't")))v*=-.74;s+=v});return s/Math.sqrt(s*s+15)}
const rxs=o=>Object.entries(o).map(([k,a])=>[k,new RegExp('\\b('+a.slice().sort((x,y)=>y.length-x.length).join('|')+')\\b','i')]);
function tryIt(){const t=$('#tx').value,cls=t.split(/(?<=[.!?])\s+|;|\bbut\b|\bhowever\b|\balthough\b/i).map(x=>x.trim()).filter(Boolean),out=[];
 cls.forEach(c=>[...rxs(D.dishes),...rxs(D.moments)].forEach(([k,r])=>{if(r.test(c))out.push([k,sc(c),c])}));const ov=sc(t);
 $('#tr').innerHTML=`<div class="card">A single overall score says <b>${ov>=0?'+':''}${ov.toFixed(2)}</b>, which hides the detail. DishRadar splits it into:</div>`+(out.map(o=>`<div class="q ${o[1]>.05?'p':''}"><b>${o[0]}</b> · ${o[1]>=0?'+':''}${o[1].toFixed(2)}<br><span class="mut">“${o[2]}”</span></div>`).join('')||'<div class="card">No dish or service moment found in this text.</div>')}

const MN=['Jan','Feb','Mar','Apr','May','Jun','Jul','Aug','Sep','Oct','Nov','Dec'];
const nxt=(lab,h)=>{const[m,y]=lab.split(' ');let k=MN.indexOf(m)+h,yy=+y+Math.floor(k/12);return MN[k%12]+' '+yy};
const fill=s=>{const o=s.slice(),ix=o.map((v,i)=>v===null?-1:i).filter(i=>i>=0);if(!ix.length)return o.map(()=>0);
 return o.map((v,i)=>{if(v!==null)return v;const p=ix.filter(j=>j<i).pop(),q=ix.find(j=>j>i);if(p===undefined)return o[q];if(q===undefined)return o[p];return o[p]+(o[q]-o[p])*(i-p)/(q-p)})};
function holtFit(y){let best=null;[.2,.4,.6].forEach(al=>[.1,.3].forEach(be=>[.8,.92].forEach(ph=>{let l=y[0],b=y[1]-y[0],sse=0;for(let t=1;t<y.length;t++){const p=l+ph*b,e=y[t]-p;sse+=e*e;const nl=al*y[t]+(1-al)*p;b=be*(nl-l)+(1-be)*ph*b;l=nl}
 if(!best||sse<best.sse)best={sse,l,b,ph}})));return best}
const M3={
 holt:(y,h)=>{const f=holtFit(y);let s=0;for(let k=1;k<=h;k++)s+=Math.pow(f.ph,k);return f.l+f.b*s},
 trend:(y,h)=>{const w=y.slice(-6),n=w.length,mx=(n-1)/2,my=mean(w),sl=w.reduce((a,v,i)=>a+(i-mx)*(v-my),0)/w.reduce((a,v,i)=>a+(i-mx)**2,0)*(n/(n+3));return mean(y.slice(-3))+sl*(h+1)*.8},
 level:(y,h)=>{let p=y[0];y.forEach(v=>p=.5*v+.5*p);return .7*p+.3*mean(y.slice(0,6))}};
function model(sr){const y=fill(sr);if(y.length<8)return null;const names=Object.keys(M3),err={naive:[]},res=[];names.forEach(n=>err[n]=[]);
 for(let t=6;t<y.length;t++){const tr=y.slice(0,t);err.naive.push(y[t]-tr[tr.length-1]);names.forEach(n=>err[n].push(y[t]-M3[n](tr,1)))}
 const w={};let sw=0;names.forEach(n=>{w[n]=1/(mean(err[n].map(e=>e*e))+1e-3);sw+=w[n]});names.forEach(n=>w[n]/=sw);
 const ens=[];for(let t=6;t<y.length;t++){const tr=y.slice(0,t);ens.push(y[t]-names.reduce((s,n)=>s+w[n]*M3[n](tr,1),0))}
 const sg=Math.max(Math.sqrt(mean(ens.map(e=>e*e))),.04);
 return{w,sg,mae:mean(ens.map(Math.abs)),nmae:mean(err.naive.map(Math.abs)),n:ens.length,f:h=>names.reduce((s,n)=>s+w[n]*M3[n](y,h),0),last:y[y.length-1]}}
const Phi=x=>{const t=1/(1+.2316419*Math.abs(x)),d=.3989423*Math.exp(-x*x/2),p=d*t*(.3193815+t*(-.3565638+t*(1.781478+t*(-1.821256+t*1.330274))));return x>0?1-p:p};
function calib(i){const st=D.stars.slice(0,i+1),av=st.map((_,j)=>{const v=D.aspects.filter(a=>a.n[j]>=mn).map(a=>a.score[j]);return v.length?mean(v):null}),ix=av.map((v,j)=>v===null?-1:j).filter(j=>j>=0);
 let b=2;if(ix.length>=5){const mx=mean(ix.map(j=>av[j])),my=mean(ix.map(j=>st[j])),vx=ix.reduce((s,j)=>s+(av[j]-mx)**2,0);if(vx>1e-6)b=Math.min(3,Math.max(1,ix.reduce((s,j)=>s+(av[j]-mx)*(st[j]-my),0)/vx))}
 const mxa=ix.length?mean(ix.map(j=>av[j])):0;return{b,a:mean(st)-b*mxa}}
let FC=null;
function fc(a,i,r){const sr=a.s.slice(0,i+1),m=model(sr);const el=$('#pm');$('#aiout').innerHTML='';
 if(!m||i+1<B+K){el.innerHTML='<div class="card">The AI model needs at least 9 months of history. Move the slider forward.</div>';mk('c2p',{type:'line',data:{labels:[],datasets:[]}});FC=null;return}
 const c=calib(i),st=s=>Math.max(1,Math.min(5,c.a+c.b*s)),thr=r.mu!==undefined?r.mu-z*r.sg:null,rows=[];
 for(let h=1;h<=3;h++){const f=m.f(h),sd=m.sg*Math.sqrt(h),lab=nxt(M[i],h),act=i+h<M.length?a.s[i+h]:null;
  rows.push({h,lab,f,lo:f-1.28*sd,hi:f+1.28*sd,star:st(f),slo:st(f-1.28*sd),shi:st(f+1.28*sd),risk:thr===null?null:Phi((thr-f)/sd),act,astar:act===null?null:st(act)})}
 const gain=m.nmae>0?(1-m.mae/m.nmae)*100:0,hist=sr.map(v=>v===null?null:st(v)),pad=Array(3).fill(null);
 FC={a,i,rows,m,gain,c};
 el.innerHTML=`<div class="kpis"><div class="kpi"><small>Predicted ★ next month (${rows[0].lab})</small><b>${rows[0].star.toFixed(2)}★</b><small>likely ${rows[0].slo.toFixed(1)} to ${rows[0].shi.toFixed(1)}</small></div><div class="kpi"><small>Predicted ★ in 3 months</small><b>${rows[2].star.toFixed(2)}★</b><small>likely ${rows[2].slo.toFixed(1)} to ${rows[2].shi.toFixed(1)}</small></div><div class="kpi"><small>Chance of dish alert in 3 months</small><b style="color:${rows[2].risk>.5?cs('bad'):rows[2].risk>.2?cs('warn'):cs('good')}">${rows[2].risk===null?'—':Math.round(rows[2].risk*100)+'%'}</b></div><div class="kpi"><small>Backtest accuracy</small><b>${gain>=0?'+':''}${gain.toFixed(0)}%</b><small>vs "same as last month" (${m.n} tests)</small></div></div>
 <div class="card ${gain>0?'good':''}"><b>How accurate is it?</b> Walking forward through this dish's own history, the model's average miss was <b>${m.mae.toFixed(3)}</b> sentiment points (about ${(m.mae*c.b).toFixed(2)}★), against <b>${m.nmae.toFixed(3)}</b> for guessing "same as last month". ${gain>0?'It beats the simple guess.':'It does not beat the simple guess for this dish, so treat the forecast with care.'} Model mix: ${Object.entries(m.w).map(([k,v])=>({holt:'damped trend',trend:'recent slope',level:'smoothed level'}[k])+' '+Math.round(v*100)+'%').join(', ')}.</div>
 <div class="tw"><table><tr><th>Month</th><th>Predicted ★</th><th>Likely range (80%)</th><th>Alert risk</th><th>What actually happened</th></tr>${rows.map(x=>`<tr><td><b>${x.lab}</b></td><td>${x.star.toFixed(2)}★</td><td>${x.slo.toFixed(2)} to ${x.shi.toFixed(2)}★</td><td>${x.risk===null?'—':Math.round(x.risk*100)+'%'}</td><td>${x.act===null?'<span class="mut">not yet known</span>':x.astar.toFixed(2)+'★ '+(x.astar>=x.slo-.001&&x.astar<=x.shi+.001?'✅ inside range':'⚠ outside range')}</td></tr>`).join('')}</table></div>
 <p class="mut">★ here is the star rating guests who mention this dish are likely to give, translated from sentiment using this restaurant's own history up to ${M[i]}. Only data up to that month is used.</p>`;
 mk('c2p',{type:'line',data:{labels:M.slice(0,i+1).concat(rows.map(x=>x.lab)),datasets:[
  {label:'Likely range (80%)',data:Array(i).fill(null).concat([hist[i]],rows.map(x=>x.shi)),borderWidth:0,pointRadius:0,backgroundColor:cs('info')+'33',fill:'+1'},
  {label:'lo',data:Array(i).fill(null).concat([hist[i]],rows.map(x=>x.slo)),borderWidth:0,pointRadius:0},
  {label:a.name+' ★ so far',data:hist.concat(pad),borderColor:cs('info'),backgroundColor:cs('info'),borderWidth:3,pointRadius:3},
  {label:'AI forecast',data:Array(i).fill(null).concat([hist[i]],rows.map(x=>x.star)),borderColor:cs('bad'),backgroundColor:cs('bad'),borderDash:[6,4],borderWidth:3,pointRadius:4}]},
  options:{animation:false,plugins:{legend:{labels:{filter:x=>x.text!=='lo'}},title:{display:true,text:a.name+': predicted rating',color:cs('ink')}},scales:{y:{title:{display:true,text:'Predicted ★'},grid:{color:cs('line')}},x:{grid:{color:cs('line')}}}}})}
const useClaude=async n=>{try{return window.claude?await claude.use(n):null}catch(e){return null}};
function localNote(F){const x=F.rows,d=x[2].f-F.m.last,cp=(D.complaints[F.a.name]||{r:{}}).r,w=Object.keys(cp).slice(0,3).join(', ');
 return `${F.a.name} is expected to ${d<-.05?'keep slipping':d>.05?'recover':'hold steady'}: about ${x[0].star.toFixed(1)}★ next month and ${x[2].star.toFixed(1)}★ in three months (likely ${x[2].slo.toFixed(1)} to ${x[2].shi.toFixed(1)}★). ${x[2].risk!==null?`The chance of an alert by then is ${Math.round(x[2].risk*100)}%.`:''}${w?` Guests recently mention: ${w}.`:''} (Built-in summary, AI explanation is not available here.)`}
$('#aib').onclick=async()=>{const F=FC,o=$('#aiout');if(!F){o.innerHTML='<div class="card">Move the slider to a month with enough history first.</div>';return}
 const sample=await useClaude('sample');if(!sample){o.innerHTML=`<div class="card ai pre">${localNote(F)}</div>`;return}
 const cp=D.complaints[F.a.name]||{r:{},rq:[]},data={restaurant:D.restaurant,dish:F.a.name,asOf:M[F.i],lastStars:+(F.c.a+F.c.b*F.m.last).toFixed(2),forecast:F.rows.map(x=>({month:x.lab,stars:+x.star.toFixed(2),low:+x.slo.toFixed(2),high:+x.shi.toFixed(2),alertRisk:x.risk===null?null:+x.risk.toFixed(2)})),backtestMAE:+F.m.mae.toFixed(3),naiveMAE:+F.m.nmae.toFixed(3),recentComplaintWords:cp.r,recentQuotes:(cp.rq||[]).slice(0,3)};
 $('#ais').textContent='Asking Claude…';o.innerHTML='<div class="card ai pre" id="aiq">Thinking…</div>';
 try{await sample(`You are a restaurant analytics assistant. A statistical model forecast the star rating guests who mention a dish will give. Using only this data, write for a restaurant manager: (1) the forecast in one plain sentence, (2) the likely reasons from the complaint words and quotes, (3) how much to trust it given the backtest numbers, (4) one concrete action this week. Under 150 words, no jargon, do not invent facts.\n\n${JSON.stringify(data)}`,{onText:({text})=>{const q=$('#aiq');if(q)q.textContent=text}});$('#ais').textContent=''}
 catch(e){$('#ais').textContent='';o.innerHTML=`<div class="card ai pre">${localNote(F)}</div>`}};
// ---- Use your data
function parseCSV(t){const R=[];let r=[],f='',q=false;t=t.replace(/^\uFEFF/,'');for(let i=0;i<t.length;i++){const c=t[i];if(q){if(c==='"'){if(t[i+1]==='"'){f+='"';i++}else q=false}else f+=c}else if(c==='"')q=true;else if(c===','){r.push(f);f=''}else if(c==='\n'||c==='\r'){if(c==='\r'&&t[i+1]==='\n')i++;r.push(f);f='';if(r.some(x=>x.trim()))R.push(r);r=[]}else f+=c}
 if(f||r.length){r.push(f);if(r.some(x=>x.trim()))R.push(r)}return R}
const pd=s=>{s=(s||'').trim();let d=new Date(s);if(!isNaN(d))return d;const m=s.match(/^(\d{1,2})[\/\-.](\d{1,2})[\/\-.](\d{2,4})/);if(m){d=new Date(+(m[3].length==2?'20'+m[3]:m[3]),+m[2]-1,+m[1]);if(!isNaN(d))return d}return null};
const esc=s=>s.replace(/[.*+?^${}()|[\]\\]/g,'\\$&'),say=(h,k)=>$('#um').innerHTML=`<div class="card ${k||''}">${h}</div>`;
let UT='';
function ingest(){const rows=parseCSV(UT);if(rows.length<2)return say('Choose a CSV file first.','bad');
 const hd=rows[0].map(x=>x.trim().toLowerCase()),fi=re=>hd.findIndex(x=>re.test(x)),id=fi(/date|time|posted|when/),it=fi(/review|text|comment|content|body|feedback/),ir=fi(/rating|stars?|score/);
 if(id<0||it<0||ir<0)return say(`Could not find all columns. Found headers: <code>${hd.join(', ')}</code>. Please include a date, a review text and a rating column.`,'bad');
 const dishes=$('#ud').value.split(/[,\n]/).map(x=>x.trim()).filter(Boolean);if(!dishes.length)return say('Add at least one dish to track (step 3).','bad');
 const rv=rows.slice(1).map(r=>({d:pd(r[id]),t:(r[it]||'').trim(),s:parseFloat(r[ir])})).filter(x=>x.d&&x.t&&x.s>=1&&x.s<=5);
 if(rv.length<50)return say(`Only ${rv.length} usable reviews found. At least 50 are needed. Check the date format and that ratings are 1 to 5.`,'bad');
 const k=d=>d.getFullYear()*12+d.getMonth(),k0=Math.min(...rv.map(x=>k(x.d))),k1=Math.max(...rv.map(x=>k(x.d))),nm=k1-k0+1;
 if(nm<9)return say(`Your data spans ${nm} months. DishRadar needs at least 9 (6 to learn the normal level and 3 to compare).`,'bad');
 const months=Array.from({length:nm},(_,i)=>MN[(k0+i)%12]+' '+Math.floor((k0+i)/12)),dm={},mo=D0.moments;dishes.forEach(x=>dm[x]=[x.toLowerCase()]);
 const rx=[...rxs(dm),...rxs(mo)],kind=n=>dm[n]?'Dish':'Moment',A={},sa=Array(nm).fill(0),sn=Array(nm).fill(0);
 rv.forEach(v=>{const m=k(v.d)-k0;sa[m]+=v.s;sn[m]++;v.t.split(/(?<=[.!?])\s+|;|\bbut\b|\bhowever\b|\balthough\b/i).map(x=>x.trim()).filter(Boolean).forEach(c=>rx.forEach(([n,r])=>{if(!r.test(c))return;const a=A[n]||(A[n]={name:n,kind:kind(n),n:Array(nm).fill(0),sum:Array(nm).fill(0),pos:Array(nm).fill(0),neg:Array(nm).fill(0),W:{},E:{},rq:[],eq:[]}),s=sc(c);
  a.n[m]++;a.sum[m]+=s;if(s>.05)a.pos[m]++;if(s<-.05)a.neg[m]++;
  if(s<-.05){c.toLowerCase().match(/[a-z']+/g).filter(w=>NW[w]).forEach(w=>{if(m>=nm-K)a.W[w]=(a.W[w]||0)+1;if(m<B)a.E[w]=(a.E[w]||0)+1});if(m>=nm-K&&a.rq.length<3)a.rq.push(c)}
  if(s>.2&&m<B&&a.eq.length<2)a.eq.push(c)}))});
 const asp=Object.values(A).filter(a=>a.n.reduce((x,y)=>x+y,0)>=10).map(a=>({name:a.name,kind:a.kind,n:a.n,score:a.sum.map((s,i)=>a.n[i]?+(s/a.n[i]).toFixed(3):0),pos:a.pos,neg:a.neg}));
 if(!asp.length)return say('None of your dishes or service words were found in the reviews. Check the spelling of the dish names.','bad');
 let ls=0;const stars=sa.map((s,i)=>sn[i]?(ls=s/sn[i]):ls),cm={};Object.values(A).forEach(a=>cm[a.name]={r:a.W,e:a.E,rq:a.rq,eq:a.eq});
 D={restaurant:$('#un').value.trim()||'Your restaurant',months,stars,nReviews:rv.length,overall:mean(rv.map(x=>x.s)).toFixed(1),aspects:asp,complaints:cm,dishes:dm,moments:mo};M=months;tmI=months.length-1;done()}
function done(){$('#sub').textContent=D.restaurant+': the overall rating says '+D.overall+' stars. This shows which dish is quietly slipping, and since when.';$('#dn').textContent=D0.nReviews.toLocaleString();$('#tm').max=M.length-1;$('#tm').value=tmI}
$('#run').onclick=()=>{ingest();if(D!==D0&&$('#um').textContent.indexOf('Could not')<0&&!$('#um .bad')){say(`✅ Loaded <b>${D.nReviews.toLocaleString()}</b> reviews for <b>${D.restaurant}</b> across ${M.length} months, tracking ${D.aspects.length} dishes and moments. Opening the Radar…`,'good');setTimeout(()=>document.querySelector('#tabs button').click(),700)}};
$('#rs').onclick=()=>{D=D0;M=D.months;tmI=M.length-1;done();say('Back to the demo restaurant.','good');renderAll()};
const rf=f=>{if(!f)return;$('#ufn').textContent=f.name;const r=new FileReader();r.onload=()=>{UT=r.result;if(!$('#un').value)$('#un').value=f.name.replace(/\.csv$/i,'').replace(/[_-]+/g,' ')};r.readAsText(f)};
$('#uf').onchange=e=>rf(e.target.files[0]);const dz=$('#dz');dz.ondragover=e=>{e.preventDefault();dz.classList.add('on')};dz.ondragleave=()=>dz.classList.remove('on');dz.ondrop=e=>{e.preventDefault();dz.classList.remove('on');rf(e.dataTransfer.files[0])};
$('#sg').onclick=async()=>{if(!UT)return $('#sgs').textContent='Choose a CSV first.';const rows=parseCSV(UT),hd=rows[0].map(x=>x.trim().toLowerCase()),it=hd.findIndex(x=>/review|text|comment|content|body|feedback/.test(x));if(it<0)return $('#sgs').textContent='No review column found.';
 const smp=rows.slice(1).map(r=>r[it]).filter(Boolean),pick=smp.filter((_,i)=>i%Math.max(1,Math.floor(smp.length/60))===0).slice(0,60),sample=await useClaude('sample');
 if(sample){$('#sgs').textContent='Asking Claude…';try{const j=await sample.json(`From these restaurant reviews, list the 6 to 12 most frequently mentioned menu items (dishes or drinks). Return only a JSON array of lowercase strings.\n\n${JSON.stringify(pick)}`);$('#ud').value=(Array.isArray(j)?j:[]).join(', ');$('#sgs').textContent='Edit the list if needed.';return}catch(e){}}
 const cnt={},stop=new Set('the this that was and but very food place which with for are had our have not they were from just good great'.split(' '));smp.forEach(t=>{const w=(t.toLowerCase().match(/[a-z]+/g)||[]);for(let i=0;i<w.length-1;i++)if(!stop.has(w[i])&&!stop.has(w[i+1])&&w[i].length>3&&w[i+1].length>3){const b=w[i]+' '+w[i+1];cnt[b]=(cnt[b]||0)+1}});
 $('#ud').value=Object.entries(cnt).sort((a,b)=>b[1]-a[1]).slice(0,8).map(x=>x[0]).join(', ');$('#sgs').textContent='Rough guess from word pairs. Please edit the list.'};
useClaude('downloads').then(d=>{if(d){$('#tpl').hidden=false;$('#tpl').onclick=()=>d.save({filename:'dishradar_template.csv',data:'date,review,rating\n2025-01-14,"The biryani was delicious but service was slow.",4\n2025-01-20,"Lovely ambience and friendly staff.",5\n'}).catch(()=>{})}});
const T=['🚨 Radar','🔍 Dish deep-dive','⏱ Time machine','🧪 Try a review','📤 Use your data'];
$('#tabs').innerHTML=T.map((t,i)=>`<button role="tab" aria-selected="${i==0}" data-i="${i}">${t}</button>`).join('');
$('#tabs').onclick=e=>{const b=e.target.closest('button');if(!b)return;document.querySelectorAll('#tabs button').forEach(x=>x.setAttribute('aria-selected',x===b));
 for(let i=0;i<5;i++)$('#p'+i).hidden=i!=b.dataset.i;renderAll()};
function opts(){const nm=R.filter(r=>r.r.st!=='Not enough data').map(r=>r.name),best=(R.filter(r=>r.lead!==null).sort((a,b)=>b.lead-a.lead)[0]||{}).name||nm[0];
 ['dd1','dd2'].forEach(id=>{const s=$('#'+id),c=s.value;s.innerHTML=nm.map(n=>`<option>${n}</option>`).join('');s.value=nm.includes(c)?c:best})}
function renderAll(){compute();opts();$('#zv').textContent=z.toFixed(1);$('#nv').textContent=mn;$('#dv').textContent=dr.toFixed(2);$('#tm').max=M.length-1;
 const v=i=>!$('#p'+i).hidden;if(v(0))radar();if(v(1))deep();if(v(2))tm()}
const D0=D;
$('#z').oninput=e=>{z=+e.target.value;renderAll()};$('#mn').oninput=e=>{mn=+e.target.value;renderAll()};$('#dr').oninput=e=>{dr=+e.target.value;renderAll()};
$('#dd1').onchange=deep;$('#dd2').onchange=tm;$('#tm').oninput=e=>{tmI=+e.target.value;tm()};$('#go').onclick=tryIt;
$('#pl').onclick=()=>{if(timer){clearInterval(timer);timer=null;$('#pl').textContent='▶ Play';return}if(tmI>=M.length-1)tmI=0;$('#pl').textContent='⏸ Pause';
 timer=setInterval(()=>{tmI++;$('#tm').value=tmI;tm();if(tmI>=M.length-1){clearInterval(timer);timer=null;$('#pl').textContent='▶ Play'}},900)};
$('#th').onclick=()=>{const r=document.documentElement,d=getComputedStyle(r).getPropertyValue('--bg').trim()==='#0E2229';r.dataset.theme=d?'light':'dark';renderAll()};
$('#sub').textContent=D.restaurant+': the overall rating says '+D.overall+' stars. This shows which dish is quietly slipping, and since when.';$('#dn').textContent=D.nReviews.toLocaleString();
$('#tm').value=tmI;done();renderAll();
</script></body></html>
'''

def site_payload(name="Spice Route Kitchen"):
    m, df = S["m"], S["df"]
    sc1, n1 = monthly(m, 1)
    stars = df.groupby("month").rating.mean().reindex(sc1.index)
    cut = lambda ser: ser.reindex(sc1.index).fillna(0).astype(int).tolist()
    aspects, comp = [], {}
    for a in sc1.columns:
        ev = m[m.aspect == a]
        aspects.append(dict(name=a, kind=ev.kind.iloc[0], n=n1[a].astype(int).tolist(),
                            score=[None if pd.isna(v) else round(float(v), 3) for v in sc1[a]],
                            pos=cut(ev[ev.sentiment > .05].groupby("month").size()),
                            neg=cut(ev[ev.sentiment < -.05].groupby("month").size())))
        neg = ev[ev.sentiment < -.05]
        rec, early = neg[neg.month >= sc1.index[-3]], neg[neg.month < sc1.index[6]]
        count = lambda d: {k: len(re.findall(rf"\b{k}\b", " ".join(d.clause.str.lower()))) for k in NEG_WORDS}
        cr, ce = count(rec), count(early)
        top = {k: v for k, v in sorted(cr.items(), key=lambda x: -x[1])[:5] if v}
        comp[a] = dict(r=top, e={k: ce[k] for k in top},
                       rq=rec.sort_values("sentiment").clause.drop_duplicates().head(3).tolist(),
                       eq=ev[(ev.sentiment > .3) & (ev.month < sc1.index[6])].clause.drop_duplicates().head(2).tolist())
    return dict(restaurant=name, months=[mon(t) for t in sc1.index], stars=[round(float(x), 3) for x in stars],
                nReviews=int(len(df)), overall=f"{df.rating.mean():.1f}", aspects=aspects, complaints=comp,
                dishes=MENU, moments=MOMENTS)

payload = site_payload()
site = SITE_TEMPLATE.replace("/*DATA*/null", json.dumps(payload).replace("<", "\\u003c"))
open("dishradar_site.html", "w", encoding="utf-8").write(site)
print(f"Website saved: dishradar_site.html ({len(site) / 1024:.0f} KB)")
try:
    from google.colab import files
    files.download("dishradar_site.html")
except Exception:
    pass

Website saved: dishradar_site.html (50 KB)


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
# Public link (needs internet; Gradio prints the URL below). Keep this notebook running while people use it.
%pip install -q gradio
import gradio as gr
frame = f'<iframe srcdoc="{html.escape(site, quote=True)}" style="width:100%;height:94vh;border:0;border-radius:12px"></iframe>'
with gr.Blocks(title="DishRadar") as demo:
    gr.HTML(frame)
demo.launch(share=True, prevent_thread_lock=True)

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://b659a55e218d497d64.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
